In [ ]:
import pandas as pd
import os 
import json

In [ ]:
pd.options.display.max_rows = None
pd.options.display.max_colwidth = None
pd.options.display.max_columns = None

In [ ]:
with open('benefit_info_20250705.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

In [ ]:
os.getcwd()

In [ ]:
df = pd.json_normalize(data)

In [ ]:
# chargeDiscountBenefit

charge_df = df[df['chargeDiscountBenefit'].notnull()][['benefitInformation.pmBenefitCode','benefitInformation.benefitName', 'chargeDiscountBenefit']]
charge_df = charge_df.reset_index(drop=True)
charge_df = charge_df.explode('chargeDiscountBenefit')
charge_df = charge_df.reset_index(drop=True)

result_df = pd.DataFrame()
for i in range(len(charge_df)):
    tmp_df = pd.DataFrame()
    tmp_df = pd.DataFrame(charge_df['chargeDiscountBenefit'][i]['productRequiredForDiscount'])
    tmp_df['benefitInformation.pmBenefitCode'] = charge_df['benefitInformation.pmBenefitCode'][i]
    tmp_df['benefitInformation.benefitName'] = charge_df['benefitInformation.benefitName'][i]
    tmp_df['discountType'] = charge_df['chargeDiscountBenefit'][i]['discountType']
    tmp_df['discountAmount'] = charge_df['chargeDiscountBenefit'][i]['discountAmount']
    tmp_df['appliedDiscountProduct'] = str(charge_df['chargeDiscountBenefit'][i]['appliedDiscountProduct'])
    try:
        tmp_df['maxDiscountAmount'] = charge_df['chargeDiscountBenefit'][i]['maxDiscountAmount']
    except:
        tmp_df['maxDiscountAmount'] = ''
    result_df = pd.concat([result_df, tmp_df], axis=0, ignore_index=True)
result_df = result_df[['benefitInformation.pmBenefitCode', 'benefitInformation.benefitName', 'appliedDiscountProduct', 'product', 'pmProductId', 'legacyProductId', 'discountType', 'discountAmount', 'maxDiscountAmount']]
charge_df = result_df

In [ ]:
# serviceContentsBenefit

service_df = df[df['serviceContentsBenefit'].notnull()][['benefitInformation.pmBenefitCode','benefitInformation.benefitName', 'serviceContentsBenefit']]
service_df = service_df.reset_index(drop=True)
service_df = service_df.explode('serviceContentsBenefit')
service_df = service_df.reset_index(drop=True)

result_df = pd.DataFrame()
for i in range(len(service_df)):
    tmp_df = pd.DataFrame()
    tmp_df = pd.DataFrame(service_df['serviceContentsBenefit'][i]['productRequiredForDiscount'])
    tmp_df['benefitInformation.pmBenefitCode'] = service_df['benefitInformation.pmBenefitCode'][i]
    tmp_df['benefitInformation.benefitName'] = service_df['benefitInformation.benefitName'][i]
    result_df = pd.concat([result_df, tmp_df], axis=0, ignore_index=True)
result_df = result_df[['benefitInformation.pmBenefitCode', 'benefitInformation.benefitName', 'product', 'pmProductId', 'legacyProductId']]
service_df = result_df

In [ ]:
# subscriptionDiscountBenefit
subs_df = df[df['subscriptionDiscountBenefit'].notnull()][['benefitInformation.pmBenefitCode','benefitInformation.benefitName', 'subscriptionDiscountBenefit']]
subs_df = subs_df.reset_index(drop=True)
subs_df = subs_df.explode('subscriptionDiscountBenefit')
subs_df = subs_df.reset_index(drop=True)

result_df = pd.DataFrame()
for i in range(len(subs_df)):
    tmp_df = pd.DataFrame()
    tmp_df = pd.DataFrame(subs_df['subscriptionDiscountBenefit'][i]['productRequiredForDiscount'])
    tmp_df['benefitInformation.pmBenefitCode'] = subs_df['benefitInformation.pmBenefitCode'][i]
    tmp_df['benefitInformation.benefitName'] = subs_df['benefitInformation.benefitName'][i]
    tmp_df['discountType'] = subs_df['subscriptionDiscountBenefit'][i]['discountType']
    tmp_df['discountAmount'] = subs_df['subscriptionDiscountBenefit'][i]['discountAmount']
    try :
        tmp_df['appliedSubscriptionProduct'] = str(subs_df['subscriptionDiscountBenefit'][i]['appliedSubscriptionProduct'])
    except:
        tmp_df['appliedSubscriptionProduct'] = str(subs_df['subscriptionDiscountBenefit'][i]['appliedPackageDiscountProduct'])
    try:
        tmp_df['maxDiscountAmount'] = subs_df['subscriptionDiscountBenefit'][i]['maxDiscountAmount']
    except:
        tmp_df['maxDiscountAmount'] = ''
    result_df = pd.concat([result_df, tmp_df], axis=0, ignore_index=True)
result_df = result_df[['benefitInformation.pmBenefitCode', 'benefitInformation.benefitName', 'appliedSubscriptionProduct', 'product', 'pmProductId', 'legacyProductId', 'discountType', 'discountAmount', 'maxDiscountAmount']]
subs_df = result_df

# 두 유형에 맞추기 위해 컬럼 생성
service_df['appliedDiscountProduct'] = 'null'
service_df['discountType'] = 'null'
service_df['discountAmount'] = 'null'
service_df['maxDiscountAmount'] = 'null'
service_df = service_df[list(charge_df.columns)]

In [ ]:
# 타입 구분
subs_df['benefitType'] = '이용료 할인'
service_df['benefitType'] = '서비스 콘텐츠 할인'
charge_df['benefitType'] = '요금 할인'

In [ ]:
service_df.columns = ['benefitInformation.pmBenefitCode', 'benefitInformation.benefitName',
       'appliedDiscountProduct', 'product', 'pmProductId', 'legacyProductId',
       'discountType', 'discountAmount', 'maxDiscountAmount', 'benefitType']

In [ ]:
subs_df.columns = ['benefitInformation.pmBenefitCode', 'benefitInformation.benefitName',
       'appliedDiscountProduct', 'product', 'pmProductId', 'legacyProductId',
       'discountType', 'discountAmount', 'maxDiscountAmount', 'benefitType']

In [ ]:
result = pd.concat([charge_df, service_df, subs_df], axis=0, ignore_index=True)

In [ ]:
len(subs_df) + len(charge_df) + len(service_df) == len(result)

In [ ]:
df_1 = df[['benefitInformation.pmBenefitCode','benefitInformation.marketingKeyword']]

In [ ]:
fin_df = pd.merge(result, df_1, on = 'benefitInformation.pmBenefitCode', how= "left")

In [ ]:
fin_df.to_csv('20250707_benefit_preprocessing.csv', index=False)

In [ ]:
col_df.loc[col_df['eng_name'] == 'newSignupSuspension', 'kor_name'] = '신규가입중단여부'
col_df.loc[col_df['eng_name'] == 'signupPeriod', 'kor_name'] = '가입년수범위'
col_df.loc[col_df['eng_name'] == 'standardDate', 'kor_name'] = '기준시점'
col_df.loc[col_df['eng_name'] == 'standardDateIncludedDuration', 'kor_name'] = '기준일 포함 기간'

In [ ]:
# 한글명으로 맵핑 불가능한 컬럼 -> 최하위까지 갔을때 맵핑이 불가능한거고 상위로 가면 가능 
col_df[col_df['kor_name'].isnull()]

In [ ]:
col_df.loc[col_df['eng_name'] == 'welfareDiscountTypeAvailability', 'kor_name'] = '복지할인유형'

In [ ]:
len(df)

In [ ]:
['benefitInformation.benefitName', 'benefitInformation.benefitType', 'benefitInformation.marketingKeyword', 'benefitInformation.benefitLineup', 'benefitApplyConditions']